# TEC206 Intermediate Programming — Week 10
## Iterators, Generators & `itertools`

**Lecture format:** explanation → demonstration → internal mechanism → guided practice → real-world examples → challenge → self-check  
**Language:** Python 3  
**Prerequisite:** variables, collections, loops, functions, exceptions, and basic classes

---

### Why this workbook exists

Python lets us process data **one item at a time** instead of loading or calculating everything at once.

This idea appears everywhere:

- `for` loops;
- lists, tuples, dictionaries, sets, and strings;
- reading large files line by line;
- database cursors;
- streaming sensor data;
- web/API pagination;
- data-processing pipelines;
- machine-learning batches;
- combinations and permutations;
- generators.

The central idea of this lecture is:

> **Do not think only in terms of “all the data”. Think in terms of “give me the next item”.**

That is the idea behind the **iterator protocol**.

> ## How to use this notebook
>
> 1. Run the notebook from top to bottom.
> 2. Before running a code cell, **predict the output**.
> 3. Pay attention to the difference between an **iterable**, an **iterator**, and a **generator**.
> 4. When you see `next()`, ask: *what state is being remembered?*
> 5. When you see `yield`, ask: *what work is delayed until later?*
> 6. Change the examples and experiment.
>
> ### Important terminology correction
>
> Python raises **`StopIteration`**, not “StopException”, when an iterator has no more values.

# 1. Learning Outcomes

By the end of this lecture, you should be able to:

- explain the difference between an **iterable**, an **iterator**, and a **generator**;
- explain how a Python `for` loop uses `iter()` and `next()` internally;
- manually step through an iterator using `next()`;
- explain the role of `StopIteration`;
- create a custom iterator class using `__iter__()` and `__next__()`;
- explain why a `for` loop and an iterator are related but are not the same thing;
- use important tools from Python's standard-library `itertools` module;
- distinguish **infinite**, **input-dependent/terminating**, and **combinatoric** iterators;
- use `count`, `cycle`, `repeat`, `chain`, `islice`, `takewhile`, `accumulate`, `combinations`, `permutations`, and `product`;
- create generator functions using `yield`;
- create generator expressions;
- explain why generators are a special type of iterator;
- explain lazy evaluation and memory efficiency;
- use `yield from` to delegate to another iterable;
- build small streaming/data-processing pipelines;
- identify appropriate real-life uses for iterators and generators.

# 2. Start With Something Familiar: A `for` Loop

Suppose a shop keeps an inventory in a dictionary.

A dictionary is an **iterable**: Python knows how to visit its elements one at a time.

By default, iterating over a dictionary produces its **keys**.

In [ ]:
inventory = {
    "laptop": 5,
    "mouse": 20,
    "keyboard": 12,
    "monitor": 7,
}

for item in inventory:
    print(item)

If we want both the key and value, we can iterate over `inventory.items()`.

In [ ]:
for item, quantity in inventory.items():
    print(f"{item:10s} -> {quantity}")

# 3. Iterable vs Iterator

These words sound similar, but they mean different things.

| Term | Meaning | Examples |
|---|---|---|
| **Iterable** | An object that can provide an iterator | list, tuple, string, dictionary, set, file |
| **Iterator** | An object that produces one value at a time using `next()` and remembers its position | `iter([1,2,3])`, file object, many `itertools` objects |
| **Generator** | A special kind of iterator created using `yield` or a generator expression | `(x*x for x in range(5))` |

A useful mental model:

```text
ITERABLE
   │
   │ iter(...)
   ▼
ITERATOR
   │
   │ next(...)
   ▼
one value
   │
   │ next(...)
   ▼
next value
   │
   └── eventually raises StopIteration
```

### Important

An iterable is **not automatically the same object** as its iterator.

For example, a list is iterable, but a list itself is not an iterator.

In [ ]:
numbers = [10, 20, 30]

print("Is the list iterable?  ", hasattr(numbers, "__iter__"))
print("Does the list have next?", hasattr(numbers, "__next__"))

iterator = iter(numbers)

print("Iterator type:", type(iterator))
print("Iterator has next?", hasattr(iterator, "__next__"))

# 4. What a `for` Loop Is Doing Internally

When we write:

```python
for item in inventory:
    print(item)
```

Python behaves approximately like this:

```python
iterator = iter(inventory)

while True:
    try:
        item = next(iterator)
        print(item)
    except StopIteration:
        break
```

So:

1. `iter(inventory)` asks the dictionary for an iterator.
2. `next(iterator)` asks for the next element.
3. The iterator remembers where it is.
4. When nothing remains, it raises `StopIteration`.
5. The `for` loop catches that exception internally and stops cleanly.

A `for` loop therefore **uses an iterator**. It is not an alternative mechanism.

In [ ]:
inventory_iterator = iter(inventory)

print(next(inventory_iterator))
print(next(inventory_iterator))
print(next(inventory_iterator))
print(next(inventory_iterator))

try:
    print(next(inventory_iterator))
except StopIteration:
    print("No values remain -> StopIteration was raised.")

## 4.1 Re-creating a `for` Loop Manually

The following code performs the same basic job as the earlier `for` loop.

In [ ]:
inventory_iterator = iter(inventory)

while True:
    try:
        item = next(inventory_iterator)
        print("Processing:", item)
    except StopIteration:
        print("Finished.")
        break

# 5. `for` Loop vs Iterator — What Is the Difference?

This is an important interview and exam question.

| `for` loop | Iterator |
|---|---|
| A **control structure** | An **object** |
| Convenient syntax for repeated processing | Supplies one item at a time |
| Automatically calls `iter()` | Usually obtained using `iter(...)` or created directly |
| Automatically calls `next()` | We can manually call `next(iterator)` |
| Automatically handles `StopIteration` | Raises `StopIteration` when exhausted |
| Usually preferred for normal looping | Useful when we need explicit control over progress/state |

### In one sentence

> A `for` loop is convenient syntax that **drives an iterator for us**.

Use manual iterators when you need fine control over *when* the next value is requested.

# 6. Iterators Remember State

An iterator keeps track of where it is.

Notice that repeatedly calling `next()` does not restart from the beginning.

In [ ]:
letters = iter(["A", "B", "C", "D"])

print(next(letters))  # A
print(next(letters))  # B

print("We can do other work here...")

print(next(letters))  # C
print(next(letters))  # D

## 6.1 An Iterator Is Usually Consumed Once

After an iterator is exhausted, calling `next()` again does not restart it.

To iterate again, we normally create a **new iterator** from the original iterable.

In [ ]:
values = [1, 2, 3]

it = iter(values)
print(list(it))       # consumes the iterator
print(list(it))       # now empty

new_it = iter(values)
print(list(new_it))   # fresh iterator

# 7. Creating Our Own Iterator Class

To create a custom iterator, a class normally implements:

- `__iter__()` — returns the iterator object;
- `__next__()` — returns the next value or raises `StopIteration`.

Example: an iterator that counts upward from `start` to `stop`.

In [ ]:
class CountUp:
    def __init__(self, start, stop):
        self.current = start
        self.stop = stop

    def __iter__(self):
        return self

    def __next__(self):
        if self.current > self.stop:
            raise StopIteration

        value = self.current
        self.current += 1
        return value


counter = CountUp(3, 7)

print(next(counter))
print(next(counter))

print("Continue using a for loop:")
for value in counter:
    print(value)

### What state is stored?

The object stores:

- the current number;
- the stopping point.

Each call to `next()` updates `self.current`.

This is why iterators are useful when a process has a **current position or state**.

# 8. Why Iterators Matter in Real Programs

Iterators are practical because they let us process items **incrementally**.

Real examples include:

| Situation | Why iteration helps |
|---|---|
| Large text/log file | Read one line at a time |
| Database query | Fetch rows progressively |
| API pagination | Process one page/result batch at a time |
| Sensor stream | Handle measurements as they arrive |
| Network socket | Process messages/chunks incrementally |
| ML training | Feed batches of samples |
| Data pipeline | Transform records step by step |
| Directory traversal | Visit files one at a time |

The common pattern is:

```text
Get next item
    │
    ▼
Process it
    │
    ▼
Need another?
  │      │
 yes     no
  │      │
  └──────┘
```

# 9. Python's `itertools` Module

Python provides the standard-library module **`itertools`** for efficient iterator building blocks.

We will group them into three broad categories:

1. **Infinite iterators** — can continue indefinitely;
2. **Input-dependent / terminating iterators** — stop according to their input or a condition;
3. **Combinatoric iterators** — create products, permutations, and combinations.

Because these tools return iterators, they are usually **lazy**: results are produced when requested.

In [ ]:
import itertools

print("itertools imported successfully.")

# 10. Infinite Iterators

Infinite iterators do not have a natural endpoint.

The most important examples are:

- `itertools.count()`
- `itertools.cycle()`
- `itertools.repeat()`

An infinite iterator must normally be combined with:

- `break`;
- `itertools.islice()`;
- `takewhile()`;
- another stopping condition.

Otherwise, the loop can run forever.

## 10.1 `itertools.count()` — Generate an Arithmetic Sequence

`count(start, step)` produces values indefinitely.

Example: print even numbers from 0 to 20.

In [ ]:
from itertools import count

for number in count(start=0, step=2):
    if number > 20:
        break
    print(number)

The iterator itself is conceptually infinite:

```text
0 → 2 → 4 → 6 → 8 → 10 → ... forever
```

Our `break` statement makes **our use of it finite**.

## 10.2 Safer Sampling With `islice()`

`islice(iterator, n)` can take the first `n` values from an iterator.

This is very useful with infinite iterators.

In [ ]:
from itertools import count, islice

first_ten_even_numbers = islice(count(0, 2), 10)

print(list(first_ten_even_numbers))

## 10.3 `itertools.cycle()` — Repeat an Iterable Forever

`cycle()` repeatedly loops through the values of an iterable.

Possible uses:

- round-robin scheduling;
- rotating server choices;
- repeating UI states;
- repeating traffic-light states;
- cycling team members.

In [ ]:
from itertools import cycle, islice

traffic_lights = cycle(["RED", "GREEN", "YELLOW"])

for state in islice(traffic_lights, 8):
    print(state)

## 10.4 `itertools.repeat()` — Repeat the Same Value

`repeat(value)` repeats a value indefinitely.

`repeat(value, times)` makes it finite.

In [ ]:
from itertools import repeat

print(list(repeat("Python", 5)))

### Mini Scenario: Retry Policy

Suppose a system allows three retry attempts.

In [ ]:
for attempt_message in repeat("Trying connection...", 3):
    print(attempt_message)

# 11. Input-Dependent / Terminating Iterators

These iterators eventually stop because of:

- the length of the input;
- a slicing limit;
- a condition;
- exhaustion of one or more source iterables.

Useful examples include:

- `chain()`
- `islice()`
- `takewhile()`
- `dropwhile()`
- `accumulate()`
- `compress()`
- `filterfalse()`
- `pairwise()`
- `zip_longest()`
- `groupby()`

## 11.1 `chain()` — Treat Multiple Iterables as One Stream

In [ ]:
from itertools import chain

morning_students = ["Aisha", "Ben", "Chen"]
afternoon_students = ["Daniel", "Eva"]

all_students = chain(morning_students, afternoon_students)

for student in all_students:
    print(student)

`chain()` does not need to create one big combined list first. It visits the first iterable and then the next.

## 11.2 `takewhile()` — Continue While a Condition Is True

Suppose sensor readings are accepted until the temperature reaches 50.

In [ ]:
from itertools import takewhile

temperatures = [21, 25, 29, 34, 41, 47, 52, 55, 48]

safe_readings = takewhile(lambda x: x < 50, temperatures)

print(list(safe_readings))

Important: `takewhile()` stops at the **first failed condition**. It does not resume later.

## 11.3 `dropwhile()` — Ignore Values Until a Condition Becomes False

In [ ]:
from itertools import dropwhile

readings = [0, 0, 0, 3, 5, 7, 0, 9]

after_startup = dropwhile(lambda x: x == 0, readings)

print(list(after_startup))

## 11.4 `accumulate()` — Running Totals

This is useful for cumulative totals such as:

- sales;
- distance travelled;
- bank balance changes;
- points;
- inventory usage.

In [ ]:
from itertools import accumulate

daily_sales = [100, 250, 175, 300, 125]

running_sales = accumulate(daily_sales)

print(list(running_sales))

## 11.5 `pairwise()` — Consecutive Pairs

Useful for differences between neighbouring observations.

In [ ]:
from itertools import pairwise

temperatures = [20, 23, 21, 25, 28]

for previous, current in pairwise(temperatures):
    print(f"{previous} -> {current}: change = {current - previous:+}")

# 12. Combinatoric Iterators

Combinatoric iterators generate mathematical arrangements.

The main tools are:

- `product()` — Cartesian product;
- `permutations()` — ordered arrangements;
- `combinations()` — unordered selections without replacement;
- `combinations_with_replacement()` — unordered selections where repetition is allowed.

These functions return **iterators**.

## 12.1 `combinations()` — Choose Groups Where Order Does Not Matter

Suppose we have three fruits and want every possible pair.

In [ ]:
from itertools import combinations

fruits = ["Apple", "Banana", "Orange"]

fruit_pairs = combinations(fruits, 2)

print("Object:", fruit_pairs)
print("All pairs:", list(fruit_pairs))

For combinations:

```text
(Apple, Banana)
(Apple, Orange)
(Banana, Orange)
```

`(Apple, Banana)` and `(Banana, Apple)` are considered the **same selection**, so only one appears.

## 12.2 Combinations of Three

In [ ]:
fruits = ["Apple", "Banana", "Orange", "Mango"]

for group in combinations(fruits, 3):
    print(group)

## 12.3 `permutations()` — Order Matters

For permutations:

`("Apple", "Banana")` is different from `("Banana", "Apple")`.

In [ ]:
from itertools import permutations

fruits = ["Apple", "Banana", "Orange"]

for arrangement in permutations(fruits, 2):
    print(arrangement)

## 12.4 `product()` — Every Cross-Combination

Useful when selecting one option from each category.

Example: shirt colour × shirt size.

In [ ]:
from itertools import product

colours = ["Black", "White"]
sizes = ["S", "M", "L"]

for choice in product(colours, sizes):
    print(choice)

## 12.5 Combination vs Permutation

| Question | Use |
|---|---|
| Choose 2 students for a committee | `combinations()` |
| Decide 1st and 2nd place | `permutations()` |
| Try every browser × operating system pair | `product()` |
| Choose 3 pizza toppings where order does not matter | `combinations()` |
| Generate possible PIN orderings from selected digits | `permutations()` |

# 13. Generators

A **generator** is a special type of iterator.

Generators are designed to make iterator creation easier.

Instead of writing a full class containing:

- state variables;
- `__iter__()`;
- `__next__()`;
- `StopIteration`;

we can often write a normal-looking function and use **`yield`**.

### Critical idea

> `return` finishes a function.  
> `yield` pauses a generator and remembers its state.

# 14. Our First Generator Function

A function containing `yield` becomes a **generator function**.

In [ ]:
def simple_generator():
    yield 10
    yield 20
    yield 30


result = simple_generator()

print(type(result))
print(next(result))
print(next(result))
print(next(result))

try:
    print(next(result))
except StopIteration:
    print("Generator is exhausted.")

Notice:

- calling `simple_generator()` does **not** immediately run through all values;
- it returns a generator object;
- each `next()` resumes execution until the next `yield`;
- local state is preserved between calls;
- when the function finishes, Python raises `StopIteration`.

# 15. `yield` Pauses and Resumes

Watch the execution order carefully.

In [ ]:
def demonstration():
    print("Step 1: generator starts")
    yield "A"

    print("Step 2: resumed after first yield")
    yield "B"

    print("Step 3: resumed after second yield")
    yield "C"

    print("Step 4: generator finishes")


g = demonstration()

print("Generator created.")
print("next ->", next(g))
print("next ->", next(g))
print("next ->", next(g))

try:
    next(g)
except StopIteration:
    print("StopIteration")

# 16. Creating a Number Generator

Instead of creating a custom iterator class, we can write:

In [ ]:
def count_up(start, stop):
    current = start

    while current <= stop:
        yield current
        current += 1


for number in count_up(3, 7):
    print(number)

Compare this with the earlier `CountUp` class.

The generator version is shorter because Python manages most of the iterator protocol for us.

# 17. Iterator vs Generator

A very important relationship:

```text
Iterator
├── custom iterator object
├── itertools iterator
├── file iterator
└── Generator
    ├── generator function
    └── generator expression
```

So **every generator is an iterator**, but **not every iterator is a generator**.

| Iterator | Generator |
|---|---|
| General protocol for producing next values | A convenient way to implement that protocol |
| Can be written as a class with `__next__()` | Usually written with `yield` |
| State must often be managed explicitly | Python preserves local variables automatically |
| May be implemented in Python or C | Created by Python generator syntax |
| Can be finite or infinite | Can also be finite or infinite |
| Often lazy | Lazy by design |

## 17.1 Important Memory Correction

It is **not correct** to say:

> “An iterator must store all values first, but a generator does not.”

Many iterators are lazy and do not store everything.

For example:

```python
iter(range(1_000_000_000))
```

does not create a billion Python integers in memory.

A better distinction is:

> A **generator is one convenient way of creating a lazy iterator** using `yield` or a generator expression.

In [ ]:
huge_range_iterator = iter(range(1_000_000_000))

print(next(huge_range_iterator))
print(next(huge_range_iterator))
print(next(huge_range_iterator))

# 18. Generator Expressions

A generator expression looks similar to a list comprehension.

List comprehension:

```python
[x * x for x in range(10)]
```

Generator expression:

```python
(x * x for x in range(10))
```

The brackets make a major difference.

In [ ]:
squares_list = [x * x for x in range(10)]
squares_generator = (x * x for x in range(10))

print(type(squares_list))
print(type(squares_generator))

print("List:", squares_list)
print("Generator values:", list(squares_generator))

# 19. List Comprehension vs Generator Expression

| List comprehension | Generator expression |
|---|---|
| Uses `[...]` | Uses `(...)` |
| Produces the full list immediately | Produces values on demand |
| Stores results in memory | Usually keeps only necessary state |
| Can index results | Cannot normally index |
| Can iterate repeatedly | Usually consumed once |
| Good when all results are needed repeatedly | Good for one-pass pipelines or large streams |

In [ ]:
import sys

list_version = [x * x for x in range(100_000)]
generator_version = (x * x for x in range(100_000))

print("Approx. list object size     :", sys.getsizeof(list_version), "bytes")
print("Approx. generator object size:", sys.getsizeof(generator_version), "bytes")

`sys.getsizeof()` is only an approximate demonstration of the object itself, but it clearly illustrates that the list stores many results while the generator does not materialize them all.

# 20. Types / Patterns of Generators

For this course, it is useful to recognise several generator styles.

### 1. Generator function
Uses `yield`.

### 2. Generator expression
Compact syntax similar to a list comprehension.

### 3. Finite generator
Eventually stops.

### 4. Infinite generator
Can continue forever.

### 5. Pipeline generator
Consumes another iterable and lazily transforms/filter values.

### 6. Delegating generator
Uses `yield from` to pass through values from another iterable.

## 20.1 Finite Generator

In [ ]:
def first_five_squares():
    for number in range(1, 6):
        yield number ** 2


print(list(first_five_squares()))

## 20.2 Infinite Generator

Be careful: an infinite generator must be bounded when demonstrated.

In [ ]:
def even_numbers():
    number = 0
    while True:
        yield number
        number += 2


evens = even_numbers()

for _ in range(10):
    print(next(evens), end=" ")

## 20.3 Pipeline Generator

Suppose we receive many numbers and only want cleaned positive squares.

In [ ]:
def positive_values(values):
    for value in values:
        if value > 0:
            yield value


def square_values(values):
    for value in values:
        yield value ** 2


raw_data = [-3, 2, 0, 5, -1, 4]

pipeline = square_values(positive_values(raw_data))

print(list(pipeline))

The stages form a pipeline:

```text
raw data
   │
   ▼
positive_values(...)
   │
   ▼
square_values(...)
   │
   ▼
consumer
```

Each stage can process one item and pass it onward.

## 20.4 Delegating Generator With `yield from`

`yield from iterable` means:

> yield every value produced by this iterable.

In [ ]:
def all_students():
    morning = ["Aisha", "Ben"]
    afternoon = ["Chen", "Daniel"]

    yield from morning
    yield from afternoon


print(list(all_students()))

# 21. Real-Life Application 1 — Reading a Large File

A file object behaves like an iterator over lines.

We do **not** need:

```python
all_lines = file.readlines()
```

to process a file.

Instead:

```python
with open("large.log") as file:
    for line in file:
        ...
```

This processes the file incrementally.

In [ ]:
# Small simulation of line-by-line processing.
# In a real project, replace this list with an opened file.

log_lines = iter([
    "INFO Application started",
    "INFO User logged in",
    "ERROR Database unavailable",
    "INFO Retrying connection",
])

for line in log_lines:
    if "ERROR" in line:
        print("Important log:", line)

# 22. Real-Life Application 2 — Sensor / Streaming Data

Imagine measurements arriving continuously.

A generator can model the stream.

In [ ]:
def simulated_sensor():
    readings = [22.1, 22.4, 22.8, 23.3, 24.1, 24.8]
    for reading in readings:
        yield reading


for temperature in simulated_sensor():
    print(f"Temperature: {temperature:.1f} °C")

# 23. Real-Life Application 3 — Process Data in Batches

Batch generators are useful in:

- machine learning;
- database insertion;
- API uploads;
- image processing;
- ETL jobs.

In [ ]:
def batches(values, batch_size):
    for start in range(0, len(values), batch_size):
        yield values[start:start + batch_size]


records = list(range(1, 11))

for batch in batches(records, 3):
    print("Processing batch:", batch)

# 24. Real-Life Application 4 — Paginated API

Many APIs return a limited number of results per page.

A generator can hide pagination from the rest of the program.

In [ ]:
def fake_api_page(page_number):
    fake_pages = {
        1: ["record-1", "record-2"],
        2: ["record-3", "record-4"],
        3: ["record-5"],
        4: [],
    }
    return fake_pages.get(page_number, [])


def api_records():
    page = 1

    while True:
        records = fake_api_page(page)

        if not records:
            break

        yield from records
        page += 1


for record in api_records():
    print(record)

The caller does not need to know about page numbers. It simply asks for the next record.

That is a major design benefit of iterators and generators: they can **hide the mechanics of obtaining the next item**.

# 25. Real-Life Application 5 — Lazy Data Transformation

Suppose we have a large stream of prices.

We want to:

1. keep valid positive prices;
2. apply 10% tax;
3. format the output.

Generator expressions can create a lazy pipeline.

In [ ]:
prices = [10, -1, 20, 0, 35, 50]

valid_prices = (p for p in prices if p > 0)
with_tax = (p * 1.10 for p in valid_prices)
formatted = (f"${p:.2f}" for p in with_tax)

for price in formatted:
    print(price)

# 26. A Common Mistake: Reusing an Exhausted Generator

Generators are normally one-pass iterators.

In [ ]:
numbers = (x for x in range(5))

print("First pass :", list(numbers))
print("Second pass:", list(numbers))

If you need to repeat the computation, create the generator again:

In [ ]:
def make_numbers():
    return (x for x in range(5))

print(list(make_numbers()))
print(list(make_numbers()))

# 27. Another Common Mistake: Accidentally Materialising Everything

This is lazy:

In [ ]:
lazy_values = (x * x for x in range(1_000_000))
print(type(lazy_values))

This materialises all results into memory:

In [ ]:
small_demo = list(x * x for x in range(10))
print(small_demo)

Converting a generator to `list(...)` is not wrong. It is useful when you truly need all values.

But it removes the memory advantage of lazy iteration.

# 28. Guided Practice 1 — Manual Iterator

Given:

```python
animals = ["cat", "dog", "bird"]
```

Complete the code so that you:

1. create an iterator;
2. call `next()` three times;
3. catch the final `StopIteration`.

In [ ]:
animals = ["cat", "dog", "bird"]

# TODO: create an iterator named animal_iterator
# TODO: print each value with next()
# TODO: handle StopIteration

### Suggested solution

In [ ]:
animals = ["cat", "dog", "bird"]
animal_iterator = iter(animals)

try:
    print(next(animal_iterator))
    print(next(animal_iterator))
    print(next(animal_iterator))
    print(next(animal_iterator))
except StopIteration:
    print("No more animals.")

# 29. Guided Practice 2 — Infinite Iterator With a Safe Stop

Use `itertools.count()` to produce multiples of 5 from 0 through 50.

In [ ]:
from itertools import count

# TODO:
# for number in count(...):
#     ...

### Suggested solution

In [ ]:
from itertools import count

for number in count(0, 5):
    if number > 50:
        break
    print(number, end=" ")

# 30. Guided Practice 3 — Combinations

A project team contains:

```python
students = ["Ali", "Beth", "Carlos", "Dina"]
```

Generate every possible 2-person team.

In [ ]:
from itertools import combinations

students = ["Ali", "Beth", "Carlos", "Dina"]

# TODO

### Suggested solution

In [ ]:
from itertools import combinations

students = ["Ali", "Beth", "Carlos", "Dina"]

for team in combinations(students, 2):
    print(team)

# 31. Guided Practice 4 — Generator Function

Create a generator called `countdown(n)` that yields:

```text
n, n-1, n-2, ..., 1
```

Example:

```python
list(countdown(5))
```

should produce:

```python
[5, 4, 3, 2, 1]
```

In [ ]:
def countdown(n):
    # TODO
    pass

### Suggested solution

In [ ]:
def countdown(n):
    while n > 0:
        yield n
        n -= 1


print(list(countdown(5)))

# 32. Guided Practice 5 — Generator Expression

Using this list:

```python
scores = [42, 71, 88, 55, 93, 67]
```

Create a generator expression that produces only scores greater than or equal to 70.

In [ ]:
scores = [42, 71, 88, 55, 93, 67]

# TODO

### Suggested solution

In [ ]:
scores = [42, 71, 88, 55, 93, 67]

passing_scores = (score for score in scores if score >= 70)

print(list(passing_scores))

# 33. Challenge — Inventory Event Stream

Suppose inventory updates arrive one at a time:

```python
events = [
    ("mouse", 5),
    ("keyboard", -2),
    ("mouse", -1),
    ("monitor", 3),
    ("keyboard", 4),
]
```

Write a generator `running_inventory(events)` that yields the current inventory dictionary after every event.

Expected concept:

```text
event 1 -> {"mouse": 5}
event 2 -> {"mouse": 5, "keyboard": -2}
...
```

### Hint

You may want to yield `inventory.copy()` rather than the same mutable dictionary object every time.

In [ ]:
events = [
    ("mouse", 5),
    ("keyboard", -2),
    ("mouse", -1),
    ("monitor", 3),
    ("keyboard", 4),
]

def running_inventory(events):
    # TODO
    pass

### Suggested solution

In [ ]:
def running_inventory(events):
    inventory = {}

    for item, change in events:
        inventory[item] = inventory.get(item, 0) + change
        yield inventory.copy()


for state in running_inventory(events):
    print(state)

# 34. Challenge — Build a Streaming Pipeline

Create a pipeline that:

1. receives numbers from `range(1, 101)`;
2. keeps only even numbers;
3. squares them;
4. keeps only squared values greater than 1000;
5. prints the first five results.

Try to avoid creating unnecessary intermediate lists.

In [ ]:
# TODO: build the pipeline with generator expressions and/or itertools.islice

### Suggested solution

In [ ]:
from itertools import islice

numbers = range(1, 101)
even_numbers = (x for x in numbers if x % 2 == 0)
squared = (x ** 2 for x in even_numbers)
large_squares = (x for x in squared if x > 1000)

print(list(islice(large_squares, 5)))

# 35. When Should I Use What?

| Situation | Good choice |
|---|---|
| Simple repetition over a collection | `for` loop |
| Need explicit control over next item | iterator + `next()` |
| Need a custom stateful traversal object | custom iterator class |
| Need a simple lazy sequence | generator function |
| Need a short lazy transformation | generator expression |
| Need endless counting/repetition | `itertools.count/cycle/repeat` |
| Need combinations/permutations | `itertools` combinatoric tools |
| Need to process a huge file | file iterator / generator pipeline |
| Need all results repeatedly and dataset is manageable | list |
| Need one pass over very large data | iterator/generator |

# 36. Summary: The Big Picture

```text
                ┌──────────────────────┐
                │       ITERABLE       │
                │ list, dict, str ...  │
                └──────────┬───────────┘
                           │ iter(...)
                           ▼
                ┌──────────────────────┐
                │       ITERATOR       │
                │ remembers position   │
                └──────────┬───────────┘
                           │ next(...)
                           ▼
                     one value
                           │
                           ▼
                    StopIteration
```

A `for` loop drives that process automatically:

```text
for loop
   │
   ├── calls iter(...)
   ├── repeatedly calls next(...)
   └── catches StopIteration
```

A generator is a convenient way to create an iterator:

```text
generator function + yield
             │
             ▼
       generator object
             │
             ▼
         iterator
```

# 37. Key Takeaways

1. **Iterable** means “can provide an iterator”.
2. **Iterator** means “can provide the next value and remember progress”.
3. `iter()` obtains an iterator.
4. `next()` asks an iterator for one more value.
5. Exhaustion is signalled by **`StopIteration`**.
6. A `for` loop automatically uses the iterator protocol.
7. `itertools` provides reusable iterator building blocks.
8. Infinite iterators must be bounded using a stopping rule.
9. `combinations()` ignores order; `permutations()` cares about order.
10. A **generator is a special kind of iterator**.
11. `yield` pauses a function and preserves its local state.
12. Generator expressions are lazy alternatives to many list comprehensions.
13. Iterators and generators are ideal for large data, streams, pipelines, and one-pass processing.

# 38. End-of-Lecture Quiz — 30 Questions

This quiz covers the complete Week 10 lecture.

### Instructions

- Choose **one answer** for each question.
- Try the questions **before checking the answer key**.
- Questions 1–10 focus on iterator fundamentals.
- Questions 11–20 focus on `itertools`.
- Questions 21–30 focus on generators, lazy evaluation, and real-world applications.

---

## Part A — Iterables, Iterators & `for` Loops

### Q1. What is an **iterable**?

A. An object that can provide an iterator  
B. An object that must contain only numbers  
C. A function that always uses `yield`  
D. An exception raised by a loop  

### Q2. Which built-in function asks an iterable for an iterator?

A. `next()`  
B. `iter()`  
C. `yield()`  
D. `range()`  

### Q3. Which built-in function asks an iterator for its next value?

A. `iter()`  
B. `loop()`  
C. `next()`  
D. `continue()`  

### Q4. What exception is raised when an iterator has no more values?

A. `IteratorError`  
B. `StopException`  
C. `StopIteration`  
D. `EndOfLoop`  

### Q5. What does a Python `for` loop do internally?

A. Converts every iterable into a list  
B. Calls `iter()` and repeatedly calls `next()`  
C. Uses only numerical indexes  
D. Creates a generator for every loop  

### Q6. Which statement best describes the difference between a `for` loop and an iterator?

A. They are exactly the same object  
B. A `for` loop is a control structure that consumes values from an iterator  
C. Iterators work only inside `for` loops  
D. A `for` loop stores all values, while an iterator never stores anything  

### Q7. Consider:

```python
numbers = [10, 20, 30]
it = iter(numbers)
print(next(it))
print(next(it))
```

What is printed?

A. `10` and `10`  
B. `10` and `20`  
C. `20` and `30`  
D. `10`, `20`, and `30`  

### Q8. Why does an iterator not restart automatically after calling `next()`?

A. It remembers its current state or position  
B. Python deletes the original iterable  
C. `next()` converts it into a tuple  
D. Iterators can return only one value  

### Q9. Which methods are normally implemented by a custom iterator class?

A. `__start__()` and `__stop__()`  
B. `__iter__()` and `__next__()`  
C. `__loop__()` and `__yield__()`  
D. `__range__()` and `__index__()`  

### Q10. What should `__next__()` do when a custom iterator has no more values?

A. Return `None` forever  
B. Restart automatically  
C. Raise `StopIteration`  
D. Delete the iterator  

---

## Part B — `itertools`

### Q11. Which module contains `count()`, `cycle()`, `combinations()`, and `permutations()`?

A. `collections`  
B. `functools`  
C. `itertools`  
D. `statistics`  

### Q12. What kind of iterator does `itertools.count()` normally create?

A. Infinite iterator  
B. Dictionary iterator  
C. File iterator  
D. Recursive iterator  

### Q13. What values are produced by:

```python
from itertools import count, islice
print(list(islice(count(0, 2), 5)))
```

A. `[0, 1, 2, 3, 4]`  
B. `[0, 2, 4, 6, 8]`  
C. `[2, 4, 6, 8, 10]`  
D. `[0, 2, 4, 6, 8, 10]`  

### Q14. Why is `islice()` useful with an infinite iterator?

A. It sorts the iterator  
B. It places a finite limit on the values consumed  
C. It converts the iterator into a dictionary  
D. It causes the iterator to restart  

### Q15. What does `itertools.cycle(["A", "B"])` conceptually produce?

A. `A, B` and then stops  
B. `A, A, B, B` and then stops  
C. `A, B, A, B, A, B, ...`  
D. Only `A` forever  

### Q16. What does this return?

```python
from itertools import combinations
list(combinations(["A", "B", "C"], 2))
```

A. `[("A","B"), ("A","C"), ("B","C")]`  
B. `[("A","B"), ("B","A"), ("A","C"), ("C","A"), ("B","C"), ("C","B")]`  
C. `[("A","A"), ("B","B"), ("C","C")]`  
D. `[("A","B","C")]`  

### Q17. What is the main difference between `combinations()` and `permutations()`?

A. Combinations care about order; permutations do not  
B. Permutations care about order; combinations do not  
C. Both always return identical results  
D. Permutations work only with numbers  

### Q18. Which function is most suitable for generating every colour × size pair?

A. `product()`  
B. `repeat()`  
C. `takewhile()`  
D. `accumulate()`  

### Q19. What does `accumulate([10, 20, 30])` produce using its default addition behaviour?

A. `10, 20, 30`  
B. `10, 30, 60`  
C. `60, 30, 10`  
D. `10, 200, 6000`  

### Q20. What does `takewhile(condition, values)` do?

A. Keeps every value anywhere in the iterable that satisfies the condition  
B. Keeps values only until the condition first becomes false  
C. Removes the first value and keeps everything else  
D. Repeats matching values forever  

---

## Part C — Generators & Lazy Evaluation

### Q21. What makes a function a **generator function**?

A. It contains `print()`  
B. It contains `yield`  
C. It contains a `for` loop  
D. It returns a list  

### Q22. What is the major behavioural difference between `return` and `yield`?

A. `return` pauses a function while `yield` permanently ends it  
B. `yield` pauses execution and preserves state; `return` normally finishes the function  
C. They always behave identically  
D. `yield` works only in classes  

### Q23. Which statement is correct?

A. Every iterator is a generator  
B. Every generator is an iterator  
C. Lists are generators  
D. Dictionaries cannot be iterated  

### Q24. Which expression creates a generator expression?

A. `[x * x for x in range(5)]`  
B. `{x * x for x in range(5)}`  
C. `(x * x for x in range(5))`  
D. `x * x for x in range(5)`  

### Q25. Why can generators be useful for very large datasets?

A. They automatically compress the dataset  
B. They can produce values lazily instead of materialising all results at once  
C. They permanently store every value  
D. They make every algorithm constant-time  

### Q26. Consider:

```python
def demo():
    yield 1
    yield 2
    yield 3

g = demo()
print(next(g))
print(next(g))
```

What is printed?

A. `1` and `1`  
B. `1` and `2`  
C. `2` and `3`  
D. `1`, `2`, and `3`  

### Q27. After a generator has been fully consumed, what normally happens if you try to consume it again?

A. It automatically restarts  
B. It produces values in reverse order  
C. It remains exhausted unless a new generator is created  
D. Python converts it into a list  

### Q28. What does `yield from some_iterable` do?

A. Sorts the iterable  
B. Yields each value from the supplied iterable  
C. Converts the iterable into a dictionary  
D. Repeats only the first value  

### Q29. Which is the best approach for processing a 100 GB log file when only one pass is required?

A. Read the entire file into one list before processing  
B. Process it incrementally line by line using iteration  
C. Copy the file into several lists first  
D. Convert every line into a global variable  

### Q30. Which situation is particularly suitable for a generator pipeline?

A. Processing a continuous sensor stream through filtering and transformation stages  
B. Storing a tiny fixed constant that never changes  
C. Defining the name of a class  
D. Importing the Python interpreter

# 39. Quiz Answer Key

<details>
<summary><strong>Click to reveal the answers</strong></summary>

| Question | Answer | Main idea |
|---:|:---:|---|
| 1 | A | An iterable can provide an iterator. |
| 2 | B | `iter()` obtains an iterator. |
| 3 | C | `next()` asks for the next value. |
| 4 | C | Exhaustion is signalled by `StopIteration`. |
| 5 | B | A `for` loop uses `iter()` and `next()` internally. |
| 6 | B | The loop drives/consumes an iterator. |
| 7 | B | The iterator advances from 10 to 20. |
| 8 | A | Iterators preserve their current position/state. |
| 9 | B | Custom iterators implement `__iter__()` and `__next__()`. |
| 10 | C | Exhausted iterators raise `StopIteration`. |
| 11 | C | These tools are in `itertools`. |
| 12 | A | `count()` has no natural endpoint. |
| 13 | B | Start at 0, step by 2, take five values. |
| 14 | B | `islice()` can safely limit an infinite source. |
| 15 | C | `cycle()` repeats the input sequence indefinitely. |
| 16 | A | Combinations select unordered pairs. |
| 17 | B | Order matters for permutations, not combinations. |
| 18 | A | `product()` generates Cartesian products. |
| 19 | B | Running totals are 10, 30, 60. |
| 20 | B | It stops at the first false condition. |
| 21 | B | `yield` creates generator behaviour. |
| 22 | B | `yield` pauses and preserves state. |
| 23 | B | A generator is a special type of iterator. |
| 24 | C | Parentheses create a generator expression here. |
| 25 | B | Generators can evaluate lazily. |
| 26 | B | Successive `next()` calls resume at successive `yield`s. |
| 27 | C | A consumed generator remains exhausted. |
| 28 | B | `yield from` delegates iteration. |
| 29 | B | Line-by-line processing avoids loading the full file. |
| 30 | A | Generator pipelines are well suited to streaming transformations. |

</details>

# 40. Optional Auto-Grading

Students can enter their answers in the dictionary below using `"A"`, `"B"`, `"C"`, or `"D"`.

Example:

```python
student_answers = {
    1: "A",
    2: "B",
    ...
}
```

Then run the grading cell.

In [ ]:
# Enter your answers here.
student_answers = {
    # 1: "A",
    # 2: "B",
    # 3: "C",
    # Continue through question 30.
}

answer_key = {
    1: "A",  2: "B",  3: "C",  4: "C",  5: "B",
    6: "B",  7: "B",  8: "A",  9: "B", 10: "C",
    11: "C", 12: "A", 13: "B", 14: "B", 15: "C",
    16: "A", 17: "B", 18: "A", 19: "B", 20: "B",
    21: "B", 22: "B", 23: "B", 24: "C", 25: "B",
    26: "B", 27: "C", 28: "B", 29: "B", 30: "A",
}

if not student_answers:
    print("Enter your answers in student_answers, then run this cell again.")
else:
    correct = 0

    for question in range(1, 31):
        response = student_answers.get(question, "Not answered").upper()
        expected = answer_key[question]

        if response == expected:
            correct += 1
            status = "Correct"
        else:
            status = f"Incorrect — correct answer: {expected}"

        print(f"Q{question:02d}: {response:>12} | {status}")

    percentage = correct / 30 * 100

    print("\\n" + "=" * 45)
    print(f"Score: {correct}/30 ({percentage:.1f}%)")

    if percentage >= 85:
        print("Excellent understanding.")
    elif percentage >= 70:
        print("Good understanding. Review the questions you missed.")
    elif percentage >= 50:
        print("Developing understanding. Revisit the key examples.")
    else:
        print("Review iterables, iterators, itertools, and generators before retrying.")

# 41. Exit Ticket

Before leaving, be able to explain these three sentences in your own words:

> **A `for` loop uses an iterator.**

> **An iterator remembers where it is and returns one value at a time.**

> **A generator is a convenient, lazy way of creating an iterator.**

### Final discussion question

If you had a **100 GB log file**, would you prefer to load the entire file into a list before processing it, or process it line by line through an iterator/generator pipeline?

Explain **why**.